In [21]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


import pandas as pd
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, accuracy_score, f1_score
import numpy as np
import matplotlib.pyplot as plt

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/tumor-classification-challenge/sample_submissions.csv
/kaggle/input/tumor-classification-challenge/train_data.csv
/kaggle/input/tumor-classification-challenge/test_data.csv


In [22]:
#loading data
train = pd.read_csv("/kaggle/input/tumor-classification-challenge/train_data.csv")
test = pd.read_csv("/kaggle/input/tumor-classification-challenge/test_data.csv")

In [ ]:
#checking correlation
correlation_matrix = train.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(10, 8))  # Adjust the figure size if needed
corplot = sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', annot_kws={"size": 6.5})  # Set the label size to 8
plt.title("Correlation Matrix", fontsize=12)  # Adjust the title size
plt.show()

In [23]:
#encoding
train['diag'] = train['diagnosis'].apply(lambda x: 1 if x == "M" else 0)

In [24]:
# Split features and target
X = train.drop(columns=['diagnosis','diag', 'id'])  # Drop 'diag' and 'id' columns
y = train['diag']

In [25]:
#outliers detection and replacement
def replace_outliers_with_median(df):
    num_col = df.select_dtypes(include=[np.number]).columns
    Q1 = df[num_col].quantile(0.25)
    Q3 = df[num_col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Replace outliers with the median
    for column in df.columns:
        median = df[column].median()
        df[column] = np.where((df[column] < lower_bound[column]) | (df[column] > upper_bound[column]), 
                              median, 
                              df[column])
    return df

# Replace outliers with median in training data
X = replace_outliers_with_median(X)

In [26]:
# Set up stratified k-fold cross-validation
skf = StratifiedKFold(n_splits=5)

In [27]:
# Define the upsampling method to balance classes
oversample = RandomOverSampler(sampling_strategy=1)

In [28]:
# Define a scaler and PCA for normalization and dimensionality reduction
scaler = StandardScaler()
pca = PCA(n_components=0.95)  # Keep components that explain 95% variance

In [29]:
# Logistic Regression model with regularization
logistic_rg = LogisticRegression(solver='liblinear') 

In [30]:
# Create a pipeline: outlier handling -> upsampling -> scaling -> PCA -> logistic regression
pipeline = Pipeline(steps=[
    ('oversample', oversample),
    ('scaler', scaler),
    ('pca', pca),
    ('logistic', logistic_rg)
])

In [31]:
# Define the grid for hyperparameter tuning (penalty and mixture)
param_grid = {
    'logistic__C': np.logspace(-10, 0, 10),  # Penalty (inverse of regularization strength)
}

In [32]:
# Define metrics for evaluation
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'f1': make_scorer(f1_score)
}

In [33]:
# Perform grid search with cross-validation
grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=skf, scoring=scoring, refit='accuracy')

In [34]:
# Fit the model to the training data
grid_search.fit(X, y)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
             estimator=Pipeline(steps=[('oversample',
                                        RandomOverSampler(sampling_strategy=1)),
                                       ('scaler', StandardScaler()),
                                       ('pca', PCA(n_components=0.95)),
                                       ('logistic',
                                        LogisticRegression(solver='liblinear'))]),
             param_grid={'logistic__C': array([1.00000000e-10, 1.29154967e-09, 1.66810054e-08, 2.15443469e-07,
       2.78255940e-06, 3.59381366e-05, 4.64158883e-04, 5.99484250e-03,
       7.74263683e-02, 1.00000000e+00])},
             refit='accuracy',
             scoring={'accuracy': make_scorer(accuracy_score),
                      'f1': make_scorer(f1_score)})

In [35]:
# Collect metrics 
results = pd.DataFrame(grid_search.cv_results_)

In [36]:
# Show the best accuracy result
print(results[results['rank_test_accuracy'] == 1][['params', 'mean_test_accuracy', 'mean_test_f1']])

                                 params  mean_test_accuracy  mean_test_f1
8  {'logistic__C': 0.07742636826811278}              0.9875      0.983158


In [37]:
# After finding the best hyperparameters, refit the model with the best parameters
best_model = grid_search.best_estimator_


In [38]:
# Fit the final model on the entire training set
best_model.fit(X, y)

Pipeline(steps=[('oversample', RandomOverSampler(sampling_strategy=1)),
                ('scaler', StandardScaler()), ('pca', PCA(n_components=0.95)),
                ('logistic',
                 LogisticRegression(C=0.07742636826811278,
                                    solver='liblinear'))])

In [40]:
# Predict on the test set (using the trained model on the test set)
X_test = test.drop(columns=['id'])  # Remove 'id' column
y_pred = best_model.predict(X_test)


In [41]:
# Count the predictions 
unique, counts = np.unique(y_pred, return_counts=True)
print(dict(zip(unique, counts)))


{0: 107, 1: 62}


In [43]:
submission = pd.DataFrame({
    'id': range(401, 570),  # Assumed ID range based on your original R code
    'diagnosis': np.where(y_pred == 1, 'M', 'B')
})
submission.to_csv("/kaggle/working/submissionlr.csv",index=False)